In [1]:
import numpy as np
import pandas as pd
sr_scores = pd.Series([90, 85, 78, 92, 85], index=['Alice', 'Bob', 'Charlie', 'David', 'Eva'])
df_sales = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "product": ["Laptop", "Phone", "Tablet", "Laptop", "Phone", "Tablet", "Laptop", "Phone"],
    "price": [1000, 500, 300, 1200, 550, 320, 1100, 600],
    "quantity": [1, 2, 3, 1, 2, 1, 1, 2],
    "order_date": [
        "2024-01-01", "2024-01-02", "2024-01-03", "2024-01-04",
        "2024-01-05", "2024-01-06", "2024-01-07", "2024-01-08"
    ]
})
df_employees = pd.DataFrame({
    "employee_id": [101, 102, 103, 104, 105, 106, 107, 108],
    "name": ["Alice", "Bob", "Charlie", "David", "Eva", "Frank", "Grace", "Helen"],
    "department": ["HR", "IT", "IT", "Finance", "HR", "IT", "Finance", "HR"],
    "age": [25, 32, 29, 41, 35, 28, 45, 30],
    "salary": [50000, 70000, 65000, 90000, 62000, 72000, 88000, 58000],
    "city": ["Hanoi", "HCMC", "Hanoi", "Danang", "HCMC", "Hanoi", "Danang", "HCMC"],
    "is_active": [True, True, False, True, True, False, True, True]
})

## 1 pd.cut()
Tạo ra một categorical dựa vào series ban đầu với bins

In [2]:
bins = pd.cut(sr_scores, bins = [0,60,80,100], labels= ['C','B','A'])
print(bins)
sr_scores.groupby(bins).max()

Alice      A
Bob        A
Charlie    B
David      A
Eva        A
dtype: category
Categories (3, object): ['C' < 'B' < 'A']


C:\Users\vutun\AppData\Local\Temp\ipykernel_16188\3813115419.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sr_scores.groupby(bins).max()


C     NaN
B    78.0
A    92.0
dtype: float64

In [3]:
def complex_dk(group):
    active_salary = group['salary'].notnull().all()
    mean_salary = group['salary'].mean() > 60000
    str_salary = (group['salary'].max()- group['salary'].min()) < 10000
    return active_salary and mean_salary and str_salary

df_employees.groupby('department').filter(complex_dk)
    

,employee_id,name,department,age,salary,city,is_active
1,102,Bob,IT,32,70000,HCMC,True
2,103,Charlie,IT,29,65000,Hanoi,False
3,104,David,Finance,41,90000,Danang,True
5,106,Frank,IT,28,72000,Hanoi,False
6,107,Grace,Finance,45,88000,Danang,True


## 2.Transform

In [4]:
# df_employees['sum_salary_department'] = df_employees.groupby('department')['salary'].transform('sum')
# df_employees['salary_percent'] =( df_employees['salary']*100/df_employees['sum_salary_department']).round(1)
# df_employees
# df_employees.drop(columns = ['sum_salary_department', 'salary_percent'])
# df_employees['salary_percent'] = df_employees.groupby('department')['salary'].transform(lambda x : (x*100/x.sum()).round(1))
# df_employees

## 3. apply()

In [5]:
import pandas as pd

data = {
    'employee': ['An', 'Bình', 'Chi', 'Dũng', 'Hoa', 'Nam'],
    'department': ['IT', 'IT', 'HR', 'HR', 'IT', 'HR'],
    'salary': [1000, 1500, 800, 1200, 1100, 1300]
}
df = pd.DataFrame(data)

# Định nghĩa hàm tùy chỉnh để lấy top 2
def get_top_2(group):
    # 'group' ở đây là một DataFrame con của từng phòng ban
    return group.sort_values('salary', ascending=False).head(2)

# Áp dụng hàm cho từng nhóm
top_employees = df.groupby('department', group_keys= False).apply(get_top_2)

print(top_employees)

  employee department  salary
5      Nam         HR    1300
3     Dũng         HR    1200
1     Bình         IT    1500
4      Hoa         IT    1100


C:\Users\vutun\AppData\Local\Temp\ipykernel_16188\1197508890.py:16: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top_employees = df.groupby('department', group_keys= False).apply(get_top_2)


In [6]:
def dept_summary(group):
    return pd.Series({
        'active_count' : group['is_active'].sum(),
        'isactive_count' : (~group['is_active']).sum()
    })
    
df_employees.groupby('department',as_index= False).apply(dept_summary)

C:\Users\vutun\AppData\Local\Temp\ipykernel_16188\561214171.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_employees.groupby('department',as_index= False).apply(dept_summary)


,department,active_count,isactive_count
0,Finance,2,0
1,HR,3,0
2,IT,1,2


#tim nguoi dung da mua hang 5 ngay ke tu lan dang nhap dau tien

In [7]:
data = {
    'User': ['User1', 'User1', 'User2', 'User2', 'User2'],
    'Date': pd.to_datetime(['2023-01-01', '2023-01-05', '2023-01-02', '2023-01-03', '2023-01-10']),
    'Action': ['Login', 'Purchase', 'Login', 'Login', 'Purchase']
}
events = pd.DataFrame(data)
def dept(group):
    is_purchase5day = ((group[group['Action'] == 'Purchase']['Date'].max() - group[group['Action'] == 'Login']['Date'].min()).days <= 5)
    return pd.Series({'is_purchase5day' : is_purchase5day})

events.groupby('User', as_index= False).apply(dept)


C:\Users\vutun\AppData\Local\Temp\ipykernel_16188\3517102986.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  events.groupby('User', as_index= False).apply(dept)


,User,is_purchase5day
0,User1,True
1,User2,False


In [8]:
df_employees.assign(
    average_salary = df_employees.groupby('department')['salary'].transform('mean')
).sort_values(
    by = 'average_salary',ascending= False
).drop(
    columns = 'average_salary'
)

,employee_id,name,department,age,salary,city,is_active
3,104,David,Finance,41,90000,Danang,True
6,107,Grace,Finance,45,88000,Danang,True
2,103,Charlie,IT,29,65000,Hanoi,False
1,102,Bob,IT,32,70000,HCMC,True
5,106,Frank,IT,28,72000,Hanoi,False
0,101,Alice,HR,25,50000,Hanoi,True
4,105,Eva,HR,35,62000,HCMC,True
7,108,Helen,HR,30,58000,HCMC,True


In [9]:
result = df_employees.loc[df_employees.groupby('department')['salary'].agg(lambda x : x.idxmax())]
result.sort_values(by = 'salary', ascending = False)

,employee_id,name,department,age,salary,city,is_active
3,104,David,Finance,41,90000,Danang,True
5,106,Frank,IT,28,72000,Hanoi,False
4,105,Eva,HR,35,62000,HCMC,True
